In [6]:
import os
import librosa
import numpy as np
from multiprocessing import Pool
from tqdm import tqdm
import traceback

# -------------------------------------------------------
# CONFIG — UPDATE THESE TWO PATHS
# -------------------------------------------------------
INPUT_ROOT = "/work/NLP-mini-project/datasets/fma/fma_clean/fma_selected"
OUTPUT_ROOT = "/work/NLP-mini-project/datasets/fma/fma_clean/mels"    

SAMPLE_RATE = 32000
N_MELS = 128
WIN_LENGTH = int(0.025 * SAMPLE_RATE)  # 25ms window
HOP_LENGTH = int(0.010 * SAMPLE_RATE)  # 10ms hop


# -------------------------------------------------------
# CREATE OUTPUT FOLDER STRUCTURE (000, 001, ...)
# -------------------------------------------------------
for i in range(256):  # FMA small/medium/large use 000–255
    folder = os.path.join(OUTPUT_ROOT, f"{i:03d}")
    os.makedirs(folder, exist_ok=True)


def process_file(path_tuple):
    """ Worker function for multiprocessing. """
    in_path, out_path = path_tuple

    # Skip if already processed
    if os.path.exists(out_path):
        return f"SKIPPED {out_path}"

    try:
        # Load audio
        y, sr = librosa.load(in_path, sr=SAMPLE_RATE, mono=True)

        # Compute mel
        mel = librosa.feature.melspectrogram(
            y=y,
            sr=sr,
            n_fft=WIN_LENGTH,
            hop_length=HOP_LENGTH,
            win_length=WIN_LENGTH,
            n_mels=N_MELS,
            power=2.0
        )

        mel_db = librosa.power_to_db(mel, ref=np.max)

        # Save output
        np.save(out_path, mel_db)

        return f"OK {out_path}"

    except Exception as e:
        error_msg = f"ERROR {in_path}: {repr(e)}"
        with open(os.path.join(OUTPUT_ROOT, "errors.log"), "a") as f:
            f.write(error_msg + "\n")
            f.write(traceback.format_exc() + "\n")
        return error_msg


# -------------------------------------------------------
# GATHER ALL MP3 FILES + MAP TO OUTPUT PATH
# -------------------------------------------------------
tasks = []

for root, dirs, files in os.walk(INPUT_ROOT):
    for f in files:
        if not f.endswith(".mp3"):
            continue

        # Example: /datasets/.../fma_selected/097/097023.mp3
        track_id = f[:-4]              # "097023"
        subfolder = track_id[:3]       # "097"

        in_path = os.path.join(root, f)
        out_path = os.path.join(OUTPUT_ROOT, subfolder, track_id + ".npy")

        tasks.append((in_path, out_path))

print(f"Found {len(tasks)} audio files to process.")


# -------------------------------------------------------
# MULTIPROCESSING EXECUTION
# -------------------------------------------------------
if __name__ == "__main__":
    num_workers = os.cpu_count() - 1 or 1
    print(f"Using {num_workers} workers...")

    with Pool(num_workers) as pool:
        for result in tqdm(pool.imap_unordered(process_file, tasks),
                           total=len(tasks), ncols=100):
            pass

    print("Done! All mels saved to:", OUTPUT_ROOT)


Found 91948 audio files to process.
Using 63 workers...


  1%|▋                                                       | 1068/91948 [00:49<2:18:54, 10.90it/s]/tmp/ipykernel_2008/3879325278.py:38: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(in_path, sr=SAMPLE_RATE, mono=True)
/work/NLP-mini-project/nlp-mini-project/.venv/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
  2%|▊                                                       | 1408/91948 [01:04<1:01:34, 24.51it/s]/tmp/ipykernel_2008/3879325278.py:38: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(in_path, sr=SAMPLE_RATE, mono=True)
/tmp/ipykernel_2008/3879325278.py:38: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(in_path, sr=SAMPLE_RATE, mono=True)
/work/NLP-mini-project/nlp-m

Done! All mels saved to: /work/NLP-mini-project/datasets/fma/fma_clean/mels


In [9]:
import os

count_mp3 = sum([len(files) for r, d, files in os.walk("/work/NLP-mini-project/datasets/fma/fma_clean/fma_selected")])
count_npy = sum([len(files) for r, d, files in os.walk("/work/NLP-mini-project/datasets/fma/fma_clean/mels")])

print("MP3 files:", count_mp3)
print("Npy files:", count_npy)


MP3 files: 91948
Npy files: 91805


In [10]:
with open("/work/NLP-mini-project/datasets/fma/fma_clean/mels/errors.log") as f:
    lines = f.readlines()

print("Errors logged:", len(lines))
print("\n".join(lines[:10]))  # preview first errors


Errors logged: 5304
ERROR /work/NLP-mini-project/datasets/fma/fma_clean/fma_selected/065/065753.mp3: NoBackendError()

Traceback (most recent call last):

  File "/work/NLP-mini-project/nlp-mini-project/.venv/lib/python3.12/site-packages/librosa/core/audio.py", line 176, in load

    y, sr_native = __soundfile_load(path, offset, duration, dtype)

                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

  File "/work/NLP-mini-project/nlp-mini-project/.venv/lib/python3.12/site-packages/librosa/core/audio.py", line 209, in __soundfile_load

    context = sf.SoundFile(path)

              ^^^^^^^^^^^^^^^^^^

  File "/work/NLP-mini-project/nlp-mini-project/.venv/lib/python3.12/site-packages/soundfile.py", line 690, in __init__

    self._file = self._open(file, mode_int, closefd)



In [11]:
error_ids = set(
    line.split()[1].split("/")[-1][:-4]  # extract track ID from path
    for line in lines if "ERROR" in line
)

print("Unique failed tracks:", len(error_ids))


Unique failed tracks: 144


In [12]:
import os
import pandas as pd
from tqdm import tqdm

# ---------------------------------------------------------
# PATHS — update if needed
# ---------------------------------------------------------
ERROR_LOG = "/work/NLP-mini-project/datasets/fma/fma_clean/mels/errors.log"
METADATA_PATH = "/work/NLP-mini-project/datasets/fma/fma_clean/clean_metadata.csv"
MP3_ROOT = "/work/NLP-mini-project/datasets/fma/fma_clean/fma_selected"

# ---------------------------------------------------------
# 1. EXTRACT FAILED TRACK IDS FROM errors.log
# ---------------------------------------------------------
error_ids = set()

with open(ERROR_LOG) as f:
    for line in f:
        if "fma_selected" in line and ".mp3" in line:
            path = line.split("fma_selected/")[-1].strip()
            # Example path: "065/065753.mp3: NoBackendError()"
            mp3name = path.split(":")[0]        # "065/065753.mp3"
            tid = os.path.basename(mp3name).replace(".mp3", "")  # "065753"
            error_ids.add(tid)

error_ids = sorted(error_ids)
print(f"Found {len(error_ids)} faulty tracks.")


Found 287 faulty tracks.


In [13]:
for tid in error_ids[:10]:
    sub = tid[:3]
    mp3 = f"/work/NLP-mini-project/datasets/fma/fma_clean/fma_selected/{sub}/{tid}.mp3"
    print(os.path.exists(mp3), mp3)


True /work/NLP-mini-project/datasets/fma/fma_clean/fma_selected/001/001486.mp3
False /work/NLP-mini-project/datasets/fma/fma_clean/fma_selected/001/001486'.mp3
True /work/NLP-mini-project/datasets/fma/fma_clean/fma_selected/002/002624.mp3
False /work/NLP-mini-project/datasets/fma/fma_clean/fma_selected/002/002624'.mp3
True /work/NLP-mini-project/datasets/fma/fma_clean/fma_selected/003/003284.mp3
False /work/NLP-mini-project/datasets/fma/fma_clean/fma_selected/003/003284'.mp3
True /work/NLP-mini-project/datasets/fma/fma_clean/fma_selected/005/005574.mp3
False /work/NLP-mini-project/datasets/fma/fma_clean/fma_selected/005/005574'.mp3
True /work/NLP-mini-project/datasets/fma/fma_clean/fma_selected/008/008669.mp3
False /work/NLP-mini-project/datasets/fma/fma_clean/fma_selected/008/008669'.mp3


In [14]:
import re

clean_error_ids = set()
for line in open(ERROR_LOG):
    if "fma_selected" in line and ".mp3" in line:
        # Extract potential filename
        path = line.split("fma_selected/")[-1]
        mp3name = path.split(":")[0]

        # Extract only the digits in the filename: 001486 -> 001486
        tid = re.sub(r"[^0-9]", "", os.path.basename(mp3name))
        
        if tid:
            clean_error_ids.add(tid)

clean_error_ids = sorted(clean_error_ids)
print("Clean unique failed tracks:", len(clean_error_ids))


Clean unique failed tracks: 144


In [16]:
import os
import pandas as pd
from tqdm import tqdm
import re

# Paths
ERROR_LOG = "/work/NLP-mini-project/datasets/fma/fma_clean/mels/errors.log"
METADATA_PATH = "/work/NLP-mini-project/datasets/fma/fma_clean/clean_metadata.csv"
MP3_ROOT = "/work/NLP-mini-project/datasets/fma/fma_clean/fma_selected"

# ---- 1. Extract clean track IDs ----
clean_error_ids = set()

with open(ERROR_LOG) as f:
    for line in f:
        if "fma_selected" in line and ".mp3" in line:
            path = line.split("fma_selected/")[-1]
            mp3name = path.split(":")[0]
            tid = re.sub(r"[^0-9]", "", os.path.basename(mp3name))
            if tid:
                clean_error_ids.add(tid)

clean_error_ids = sorted(clean_error_ids)
print("Total corrupted tracks:", len(clean_error_ids))


# ---- 2. Remove from metadata ----
df = pd.read_csv(METADATA_PATH)
df["track_id"] = df["track_id"].astype(str)

filtered_df = df[~df["track_id"].isin(clean_error_ids)]
filtered_path = METADATA_PATH.replace(".csv", "_filtered.csv")
filtered_df.to_csv(filtered_path, index=False)

print("Original metadata:", len(df))
print("Filtered metadata:", len(filtered_df))
print("Saved cleaned metadata to:", filtered_path)


# ---- 3. Delete corrupted MP3s ----
deleted = 0
missing = 0

for tid in tqdm(clean_error_ids):
    sub = tid[:3]
    mp3_path = os.path.join(MP3_ROOT, sub, f"{tid}.mp3")

    if os.path.exists(mp3_path):
        os.remove(mp3_path)
        deleted += 1
    else:
        missing += 1

print("MP3 files deleted:", deleted)
print("MP3 files missing already:", missing)


Total corrupted tracks: 144
Original metadata: 91948
Filtered metadata: 91948
Saved cleaned metadata to: /work/NLP-mini-project/datasets/fma/fma_clean/clean_metadata_filtered.csv


100%|██████████| 144/144 [00:00<00:00, 8493.96it/s]

MP3 files deleted: 0
MP3 files missing already: 144
